<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/main/notebooks/gpt_2_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model-Config:

In [76]:
import torch
import torch.nn as nn
import tiktoken

# Tokenizer to be used
tokenizer = tiktoken.get_encoding("gpt2")

GPT_CONFIG_124M = {
    "vocab_size": tokenizer.n_vocab,  # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of layers
    "temperature": 1.0,  # Temperature
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": True,  # Query-Key-Value bias
}

Attention-Layer:

In [77]:
class SupremeMultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        d_k,
        context_length,
        temperature,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()
        assert d_k % num_heads == 0, "d_k must be divisible by num_heads"

        self.d_out = d_out
        self.d_k = d_k
        self.temperature = temperature
        self.num_heads = num_heads
        self.head_dim = d_k // num_heads

        self.W_key = nn.Linear(d_in, d_k, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_k, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_k, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

        self.out_proj = nn.Linear(d_k, d_out)

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / (self.head_dim**0.5 * self.temperature), dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        context_vec = context_vec.transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_k)

        context_vec = self.out_proj(context_vec)
        return context_vec

Layer-Norm:

In [78]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

GELU-Activation for Feed-Forward-Network:

In [79]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )

Feed-Forward-Network:

In [80]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

Transformer-Block:

In [81]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = SupremeMultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            d_k=cfg["emb_dim"],
            context_length=cfg["context_length"],
            temperature=cfg["temperature"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

Model:

In [82]:
class SupremeGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

Test-function:

In [110]:
import torch


def test_generate_text(model, idx, max_new_tokens, context_size, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k, dim=-1)
            min_val = top_logits[:, [-1]]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits
            )

        probas = torch.softmax(logits, dim=-1)

        if top_k is not None:
            idx_next = torch.multinomial(probas, num_samples=1)
        else:
            idx_next = torch.argmax(probas, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

Test-Data:

In [135]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

Test on untrained model:

In [136]:
untrained_model = SupremeGPTModel(GPT_CONFIG_124M)
untrained_model.eval()

out = test_generate_text(
    model=untrained_model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=50,
)

decoded_text_untrained = tokenizer.decode(out.squeeze(0).tolist())
print("Input:  ", start_context)
print("Output: ", decoded_text_untrained)

Input:   Hello, I am
Output:  Hello, I am recruitiere SylJack acts divid


Test on trained model (load GPT2 weiths to simulate trained model):

In [120]:
# Download GPT2-weights from hugging face:

import urllib.request
import os

weights_url = "https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/gpt2-small-124M.pth"
weights_path = "gpt2-small-124M.pth"

if not os.path.exists(weights_path):
    urllib.request.urlretrieve(weights_url, weights_path)

state_dict = torch.load(weights_path, map_location="cpu")

In [138]:
# Create model with the downloaded weights and test with Top-k Sampling
import textwrap

trained_model = SupremeGPTModel(GPT_CONFIG_124M)
trained_model.load_state_dict(state_dict)
trained_model.eval()

out_trained = test_generate_text(
    model=trained_model,
    idx=encoded_tensor,
    max_new_tokens=50,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=50,
)

decoded_text_trained = tokenizer.decode(out_trained.squeeze(0).tolist())
print("Input: ", start_context, "\n")
print("Output:\n" + "\n".join(textwrap.wrap(decoded_text_trained, width=80)))

Input:  Hello, I am 

Output:
Hello, I am the one who first called you today. It seems as if there is
something that's wrong with us if your parents don't believe in you. Why, then,
are you so upset that my mom and dad are so eager to hear your complaints?
